
# Rainfall Analysis and Planting Advisory

This project analyses monthly rainfall data from five Kenyan counties using Python and NumPy. The objective is to clean the Machakos rainfall data, calculate rainfall statistics, identify wettest months and dry months, and recommend suitable two-month planting windows.




In [1]:

import numpy as np
import csv

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

months = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70]
]

machakos_raw = [
    "48", "35", "88", "N/A", "140", " 20 ",
    "9", "12", "", "45", "182", "96"
]

print("Dataset loaded successfully.")
print("Number of counties:", len(counties))
print("Number of Machakos readings:", len(machakos_raw))


Dataset loaded successfully.
Number of counties: 5
Number of Machakos readings: 12



## Task 1: Cleaning Machakos Rainfall Data

The Machakos rainfall readings were manually entered and contain invalid values. A cleaning function will convert valid readings into integers and replace invalid readings with the rounded average of valid readings.


In [2]:

def clean_machakos(raw_values, month_names):
    """Clean Machakos rainfall readings and impute invalid values."""

    valid_values = []
    invalid_months = []

    for value, month in zip(raw_values, month_names):
        try:
            valid_values.append(int(value.strip()))
        except ValueError:
            invalid_months.append(month)

    average = round(sum(valid_values) / len(valid_values))

    cleaned_values = []

    for value in raw_values:
        try:
            cleaned_values.append(int(value.strip()))
        except ValueError:
            cleaned_values.append(average)

    print("Imputed months:", invalid_months)
    print("Imputation value:", average, "mm")

    return cleaned_values


machakos_clean = clean_machakos(machakos_raw, months)

print("Cleaned Machakos data:", machakos_clean)


Imputed months: ['April', 'September']
Imputation value: 68 mm
Cleaned Machakos data: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]



## Task 2: Combining County Data into a NumPy Array

The cleaned Machakos readings are combined with the rainfall data from the other four counties to create a two-dimensional NumPy array.


In [3]:

# Add the cleaned Machakos data to the other counties
all_rows = clean_rows + [machakos_clean]

# Convert the combined data into a NumPy array
R = np.array(all_rows)

# Display the array and its properties
print("Combined Rainfall Array:")
print(R)

print("\nArray shape:", R.shape)
print("Array dtype:", R.dtype)


Combined Rainfall Array:
[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]

Array shape: (5, 12)
Array dtype: int64



## Task 3: Annual Rainfall Totals and Monthly Averages

Calculate the total annual rainfall for each county and the average rainfall for each month across all five counties using NumPy axis operations.


In [4]:

# Calculate annual rainfall totals for each county
annual_totals = R.sum(axis=1)

# Calculate average rainfall for each month across all counties
monthly_averages = R.mean(axis=0).round(2)

print("Annual Rainfall Totals:")
for county, total in zip(counties, annual_totals):
    print(f"{county}: {total} mm")

print("\nMonthly Average Rainfall:")
for month, average in zip(months, monthly_averages):
    print(f"{month}: {average} mm")


Annual Rainfall Totals:
Nairobi: 1045 mm
Nakuru: 916 mm
Kisumu: 1371 mm
Mombasa: 1075 mm
Machakos: 811 mm

Monthly Average Rainfall:
January: 47.0 mm
February: 47.6 mm
March: 91.0 mm
April: 164.4 mm
May: 171.2 mm
June: 67.6 mm
July: 56.6 mm
August: 58.2 mm
September: 64.2 mm
October: 67.8 mm
November: 127.4 mm
December: 80.6 mm



## Task 4: Identifying the Wettest Month per County

Use NumPy's argmax function with axis=1 to identify the month with the highest rainfall for each county. Use zip() to pair county names with their wettest month.


In [5]:

# Find the index of the highest rainfall for each county
wettest_indices = np.argmax(R, axis=1)

# Display each county and its wettest month
print("Wettest Month per County:")

for county, index in zip(counties, wettest_indices):
    print(f"{county}: {months[index]} ({R[counties.index(county), index]} mm)")


Wettest Month per County:
Nairobi: April (219 mm)
Nakuru: April (145 mm)
Kisumu: April (240 mm)
Mombasa: May (240 mm)
Machakos: November (182 mm)



## Task 5: Counting Dry Months

A dry month is defined as a month receiving less than 50 mm of rainfall. Boolean masking is used to identify and count dry months for each county.


In [6]:

# Create a Boolean mask for rainfall below 50 mm
dry_mask = R < 50

# Count the dry months for each county
dry_month_counts = dry_mask.sum(axis=1)

print("Dry Months per County:")

for county, count in zip(counties, dry_month_counts):
    print(f"{county}: {count} dry months")


Dry Months per County:
Nairobi: 4 dry months
Nakuru: 3 dry months
Kisumu: 0 dry months
Mombasa: 2 dry months
Machakos: 6 dry months



## Task 6: Identifying the Best Planting Windows

Calculate the combined rainfall for every pair of consecutive months using NumPy slicing. Identify the two-month period with the highest combined rainfall for each county using argmax.


In [7]:

# Add rainfall from consecutive months
consecutive_totals = R[:, :-1] + R[:, 1:]

# Find the position of the highest two-month rainfall
best_window_indices = np.argmax(consecutive_totals, axis=1)

# Display the recommended planting windows
print("Recommended Planting Windows:")

for i, county in enumerate(counties):
    start = best_window_indices[i]
    window = f"{months[start]} - {months[start + 1]}"
    total = consecutive_totals[i, start]

    print(f"{county}: {window} ({total} mm)")


Recommended Planting Windows:
Nairobi: April - May (395 mm)
Nakuru: April - May (265 mm)
Kisumu: April - May (420 mm)
Mombasa: April - May (390 mm)
Machakos: November - December (278 mm)



## Task 7: Exporting the Rainfall Summary CSV

Export the rainfall analysis results into a CSV file containing one row per county. The file includes annual rainfall totals, wettest months, dry-month counts and recommended planting windows.


In [8]:

with open("rainfall_summary.csv", "w", newline="") as file:
    writer = csv.writer(file)

    # Write the CSV header
    writer.writerow([
        "county",
        "annual_total_mm",
        "wettest_month",
        "dry_months",
        "planting_window"
    ])

    # Write one row for each county
    for i, county in enumerate(counties):
        wettest_month = months[wettest_indices[i]]
        dry_months = dry_month_counts[i]

        start = best_window_indices[i]
        planting_window = f"{months[start]}-{months[start + 1]}"

        writer.writerow([
            county,
            annual_totals[i],
            wettest_month,
            dry_months,
            planting_window
        ])

print("rainfall_summary.csv created successfully!")


rainfall_summary.csv created successfully!


In [9]:

with open("rainfall_summary.csv", "r", newline="") as file:
    print(file.read())


county,annual_total_mm,wettest_month,dry_months,planting_window
Nairobi,1045,April,4,April-May
Nakuru,916,April,3,April-May
Kisumu,1371,April,0,April-May
Mombasa,1075,May,2,April-May
Machakos,811,November,6,November-December




## Planting Advisory

The NGO should prioritize April-May as the recommended two-month planting window for Nairobi, Nakuru, Kisumu and Mombasa because these months have the highest combined rainfall in those counties. For Machakos, November-December is the recommended planting window, with a combined rainfall of 278 mm. Kisumu recorded the highest annual rainfall at 1,371 mm, while Nakuru recorded the lowest at 916 mm. Farmers in Machakos should be given additional guidance because the county recorded six dry months, indicating greater rainfall variability and potential water stress. The NGO should encourage farmers to monitor local weather forecasts before planting, as historical rainfall patterns do not guarantee future rainfall. These recommendations are based on monthly rainfall totals and should be combined with information about soil conditions, crop types and water availability.



## Data Quality Notes

The Machakos rainfall dataset contained two invalid entries: "N/A" for April and an empty value for September. These entries were replaced with 68 mm, calculated from the average of the ten valid readings and rounded to the nearest integer. This imputation assumes that the missing readings can reasonably be represented by the average of the available months, which may not reflect actual rainfall conditions. The remaining four county datasets were used as supplied without additional cleaning. The analysis is based on historical monthly rainfall data and does not account for rainfall distribution within each month, soil moisture, or future weather conditions.
